# Pure Classification Baseline for Brain Tumor MRI

**Pipeline:** Raw MRI → ResNet50 Classification → Grad-CAM + Template Report

This notebook is a clean, reproducible baseline that feeds the full original MRI images directly into ResNet50 for 4-class classification. It uses **no segmentation, no cropping, and no U-Net**.

The purpose is to provide a strong baseline for comparing against the main multi-stage pipeline in `notebooks/main_pipeline.ipynb`.

## Important constraints
- Reuses the same stratified 80/10/10 train/val/test split as the main pipeline
- Reuses the existing project modules in `src/`
- Does **not** modify the original pipeline code
- Saves outputs, figures, metrics, and Grad-CAM examples in the same project folders
- Fully reproducible and ready to run top-to-bottom in Colab or locally


---
## 0.1 Environment Setup & Installation

In [ ]:
import os, sys, subprocess
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


_NEED = False
for _pkg in ['medpy', 'cv2', 'skimage']:
    try:
        __import__(_pkg)
    except (ImportError, ValueError):
        _NEED = True
        break

if _NEED:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'medpy', 'opencv-python-headless', 'scikit-image',
        'nibabel', 'albumentations',
        'grad-cam', 'captum', 'open_clip_torch', 'omegaconf', 'kaggle'
    ])
    print('Packages installed.')
else:
    print('All dependencies already installed.')

---
## 0.2 Project Root, Imports, and Reproducibility

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import importlib
import json
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

if IN_COLAB and Path('/content/drive/MyDrive/comp4471Project').exists():
    PROJECT_ROOT = Path('/content/drive/MyDrive/comp4471Project')
else:
    candidates = [Path.cwd().parent, Path.cwd()]
    PROJECT_ROOT = next((p for p in candidates if (p / 'src' / 'utils.py').exists()), None)
    if PROJECT_ROOT is None:
        raise RuntimeError('Could not locate the project root.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.utils, src.preprocessing, src.classification, src.explainability, src.visualization
for _m in [src.utils, src.preprocessing, src.classification, src.explainability, src.visualization]:
    importlib.reload(_m)

from src.utils import seed_everything, get_device, load_config
from src.preprocessing import discover_images, stratified_split, CLASS_NAMES, CLASS_TO_IDX
from src.classification import (
    BrainTumorDataset, build_resnet50, compute_class_weights,
    get_train_transforms, get_val_transforms,
    train_classifier, evaluate
)
from src.explainability import generate_gradcam, overlay_gradcam, generate_template_report
from src.visualization import plot_class_distribution, plot_confusion_matrix, plot_training_curves

SEED = 42
seed_everything(SEED)
DEVICE = get_device()
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

cfg = load_config(str(PROJECT_ROOT / 'configs' / 'pipeline_config.yaml'))
print(f"Config loaded. Classes: {cfg['dataset']['classes']}")

---
## 0.3 Configuration & Paths

In [ ]:
RAW_DATA_DIR  = PROJECT_ROOT / cfg['paths']['raw_data_dir']
PROCESSED_DIR = PROJECT_ROOT / cfg['paths']['processed_data_dir']
SEG_MODEL_DIR = PROJECT_ROOT / cfg['paths']['segmentation_model_dir']
CLS_MODEL_DIR = PROJECT_ROOT / cfg['paths']['classification_model_dir']
VLM_MODEL_DIR = PROJECT_ROOT / cfg['paths']['vlm_model_dir']
OUTPUT_DIR    = PROJECT_ROOT / cfg['paths']['output_dir']

for d in [PROCESSED_DIR, SEG_MODEL_DIR, CLS_MODEL_DIR, VLM_MODEL_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'figures').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'reports').mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Raw data dir: {RAW_DATA_DIR}')

---
# Stage 0 – Data Preparation & Exploratory Data Analysis

In [ ]:
samples = discover_images(str(RAW_DATA_DIR))
print(f'Total images found: {len(samples)}')
print(f'CLASS_NAMES: {CLASS_NAMES}')
print(f'CLASS_TO_IDX: {CLASS_TO_IDX}')
assert len(samples) > 0, f'No images found in {RAW_DATA_DIR}.'

labels = [s[1] for s in samples]
label_counts = Counter(labels)
print('\nClass distribution:')
for cls_idx, count in sorted(label_counts.items()):
    print(f'  {CLASS_NAMES[cls_idx]:>12s}: {count:>5d}')

plot_class_distribution(
    labels,
    class_names=CLASS_NAMES,
    title='Kaggle Brain Tumor MRI - Class Distribution',
    save_path=str(OUTPUT_DIR / 'figures' / 'class_distribution.png'),
)

In [ ]:
train_samples, val_samples, test_samples = stratified_split(samples, ratios=(0.8, 0.1, 0.1), seed=SEED)
print(f'Train: {len(train_samples)}  |  Val: {len(val_samples)}  |  Test: {len(test_samples)}')

for split_name, split_data in [('Train', train_samples), ('Val', val_samples), ('Test', test_samples)]:
    split_labels = [s[1] for s in split_data]
    counts = Counter(split_labels)
    print(f'\n{split_name} split:')
    for cls_idx in sorted(counts):
        print(f'  {CLASS_NAMES[cls_idx]:>12s}: {counts[cls_idx]:>5d}')

---
# Stage 1 – Pure Classification Model

In [ ]:
train_dataset = BrainTumorDataset(train_samples, transform=get_train_transforms(input_size=224, cfg=cfg.get('classification', {})))
val_dataset = BrainTumorDataset(val_samples, transform=get_val_transforms(input_size=224))
test_dataset = BrainTumorDataset(test_samples, transform=get_val_transforms(input_size=224))

train_loader = DataLoader(train_dataset, batch_size=cfg.get('classification', {}).get('batch_size', 32), shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=cfg.get('classification', {}).get('batch_size', 32), shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=cfg.get('classification', {}).get('batch_size', 32), shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}  |  Test batches: {len(test_loader)}')

model = build_resnet50(num_classes=4, pretrained=True).to(DEVICE)
class_weights = compute_class_weights([s[1] for s in train_samples]).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.get('classification', {}).get('learning_rate', 1e-4), weight_decay=cfg.get('classification', {}).get('weight_decay', 1e-4))
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

best_cls_dir = CLS_MODEL_DIR / 'pure_baseline'
best_cls_dir.mkdir(parents=True, exist_ok=True)
best_model_path = str(best_cls_dir / 'best_resnet50_pure_classification.pth')

history = train_classifier(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=DEVICE,
    epochs=cfg.get('classification', {}).get('epochs', 50),
    patience=cfg.get('classification', {}).get('patience', 10),
    save_path=best_model_path,
)

In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
test_loss, test_metrics, test_cm = evaluate(model, test_loader, criterion, DEVICE, desc='  Test')

print('\nTest metrics')
print(json.dumps(test_metrics, indent=2))
print(f'Test loss: {test_loss:.4f}')

metrics_path = best_cls_dir / 'test_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump({'test_loss': float(test_loss), **{k: float(v) for k, v in test_metrics.items()}}, f, indent=2)

plot_training_curves(history, save_path=str(OUTPUT_DIR / 'figures' / 'pure_classification_training_curves.png'))
plot_confusion_matrix(test_cm, class_names=CLASS_NAMES, title='Pure Classification Test Confusion Matrix', save_path=str(OUTPUT_DIR / 'figures' / 'pure_classification_confusion_matrix.png'))

---
# Stage 2 – Grad-CAM + Template Reports

In [ ]:
sample_paths = [s[0] for s in test_samples[:8]]
sample_labels = [s[1] for s in test_samples[:8]]

reports = []
for i, (img_path, label) in enumerate(zip(sample_paths, sample_labels), start=1):
    cam = generate_gradcam(model=model, image_path=img_path, device=DEVICE)
    overlay = overlay_gradcam(img_path, cam)
    out_path = OUTPUT_DIR / 'figures' / f'gradcam_pure_{i:02d}.png'
    overlay.save(out_path)
    report = generate_template_report(
        image_path=img_path,
        true_label=CLASS_NAMES[label],
        predicted_label=CLASS_NAMES[label],
        save_path=str(OUTPUT_DIR / 'reports' / f'report_pure_{i:02d}.txt')
    )
    reports.append(report)

print(f'Generated {len(reports)} example Grad-CAM overlays and reports.')

---
## Summary

This notebook implements the pure classification baseline using the full MRI images only.

You can now compare its metrics directly with the segmentation-first pipeline to demonstrate the value of tumor localization before classification.